# Baseline — VADER 烟雾测试版

在最少的 cell、样例数下跑通 VADER baseline，确认本机 / 服务器链路可用后再放大。

默认只取前 **10 个 `CommentBlock`**，跑完打印一个最小指标字典（accuracy + confusion matrix），并写到 `outputs/vader_smoke/`。

改全量见最后一段 `如何切到全量`。

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
VADER_DIR = PROJECT_ROOT / "baseline" / "vaderSentiment"
if str(VADER_DIR) not in sys.path:
    sys.path.insert(0, str(VADER_DIR))

from config import DEFAULT_INPUT_PATH  # noqa: E402
from data import build_comment_blocks, load_posts  # noqa: E402
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer  # noqa: E402

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DEFAULT_INPUT_PATH =", DEFAULT_INPUT_PATH)

In [ ]:
# 一行开关：True = 本地烟雾测试（取前 10 个 block），False = 跑全量
SMOKE = True
LIMIT_BLOCKS = 10 if SMOKE else None

posts = load_posts(DEFAULT_INPUT_PATH)
blocks, issues = build_comment_blocks(posts)
if LIMIT_BLOCKS is not None:
    blocks = blocks[:LIMIT_BLOCKS]

print(f"blocks={len(blocks)} issues={len(issues)} smoke={SMOKE}")

analyzer = SentimentIntensityAnalyzer()
LABEL_BULLISH, LABEL_BEARISH = 1, -1

pairs = []
rows = []
for block in blocks:
    text = block.root_comment.text or ""
    scores = analyzer.polarity_scores(text)
    compound = float(scores["compound"])
    if compound >= 0.05:
        pred = LABEL_BULLISH
    elif compound <= -0.05:
        pred = LABEL_BEARISH
    else:
        # 中性按多数类回退（烟雾测版本用 1，等全量版本会单独统计）
        pred = LABEL_BULLISH
    truth = int(block.label)
    pairs.append((truth, pred))
    rows.append({
        "block_id": block.block_id,
        "true_label": truth,
        "pred_label_int": pred,
        "compound": compound,
        "text": text[:120],
    })

n = len(pairs)
correct = sum(1 for t, p in pairs if t == p)
acc = correct / n if n else 0.0
matrix = {"bullish": {"bullish": 0, "bearish": 0}, "bearish": {"bullish": 0, "bearish": 0}}
for t, p in pairs:
    matrix["bullish" if t == 1 else "bearish"]["bullish" if p == 1 else "bearish"] += 1
print(f"n={n}  acc={acc:.3f}")
print(f"confusion_matrix={matrix}")

In [ ]:
# 抽样展示 + 保存
OUT_DIR = PROJECT_ROOT / "outputs" / "vader_smoke"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("--- 抽检前 3 条 ---")
for row in rows[:3]:
    print(
        f"- {row['block_id']} | true={row['true_label']:+d} | "
        f"pred={row['pred_label_int']:+d} | compound={row['compound']:+.3f} | "
        f"text={row['text']}"
    )

summary = {
    "config": {
        "input": str(DEFAULT_INPUT_PATH),
        "limit_blocks": LIMIT_BLOCKS,
        "smoke": SMOKE,
    },
    "total": n,
    "accuracy": acc,
    "confusion_matrix": matrix,
    "predictions": rows,
}
(OUT_DIR / "metrics.json").write_text(
    json.dumps({k: v for k, v in summary.items() if k != "predictions"}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
with (OUT_DIR / "predictions.jsonl").open("w", encoding="utf-8") as fp:
    for row in rows:
        fp.write(json.dumps(row, ensure_ascii=False) + "\n")
print("wrote", OUT_DIR / "metrics.json")
print("wrote", OUT_DIR / "predictions.jsonl")

## 如何切到全量

本地烟雾测试通过后，把第二个代码 cell 顶部的开关改一下：

```python
SMOKE = False
LIMIT_BLOCKS = None  # 跑全量
```

然后整本重跑一遍。输出会改写到 `outputs/vader/metrics.json` 与 `predictions.jsonl`；如果想保留烟雾结果输出，把 `OUT_DIR` 路径一并改掉即可。

服务器上推荐命令：

```bash
"D:/anaconda/envs/sentiment/python.exe" -m jupyter nbconvert --to notebook --execute \
    --ExecutePreprocessor.timeout=600 \
    notebooks/01_vader_baseline.ipynb \
    --output 01_vader_baseline_full.ipynb
```

把 `SMOKE = False` 改好后用 `--execute` 一次性跑完，避免在服务器上手点。